# SAML-D + IBM AML Merged Federated Learning Benchmark

MLP FL (FedAvg | FedProx | PersFL) + XGBoost (CUDA) + LightGBM (CPU) + 3 MoE Gates

**Run environment:** RAPIDS, GPU P100, Always use latest

In [1]:
!pip install lightgbm xgboost imbalanced-learn -q

## Cell 2: Imports + Config

In [2]:
# ================================================================
# MERGED DATASET: SAML-D + IBM AML (HI-Small)
# FL  : FedAvg | FedProx | FedNova | PersFL  (sklearn MLP via cuml.accel)
# ML  : XGBoost (device=cuda) + LightGBM (device=cpu)
# MoE : Static | Performance | Typology-Aware (novel)
#
# Partition strategy: DATA SOURCE partition
#   Banks 0-3 : SAML-D transactions (Dirichlet alpha=0.5 within SAML-D)
#   Banks 4-5 : IBM AML transactions (split by From Bank ID range)
#
# This creates GENUINE typology heterogeneity between bank groups:
#   SAML-D banks : cross-border wire, smurfing, structuring (28 typologies)
#   IBM AML banks: ACH, wire layering, cheque fraud, Reinvestment patterns
#
# SETUP: !pip install lightgbm xgboost imbalanced-learn -q
# ================================================================

# Import SMOTE before cuml.accel intercepts sklearn
from imblearn.over_sampling import SMOTE

import cuml
cuml.accel.install()
print("cuml.accel installed — sklearn MLP on GPU")

import os, gc, time, warnings, random
from collections import defaultdict

import numpy as np
import pandas as pd
import cupy as cp

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score, recall_score,
    average_precision_score, matthews_corrcoef,
    balanced_accuracy_score, confusion_matrix
)
from sklearn.model_selection import train_test_split

import xgboost as xgb
import lightgbm as lgb

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

print("All imports OK")

T0  = time.time()
OUT = '/kaggle/working' if os.path.exists('/kaggle') else '.'
def elapsed(): return f"{(time.time()-T0)/60:.1f}m"
def flush():
    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()

# ================================================================
# CONFIG
# ================================================================
SAML_ROWS     = 9_500_000
IBM_FILE      = 'HI-Small_Trans.csv'   # ~5M rows, ~5K fraud
N_SAML_BANKS  = 4      # SAML-D partitioned with Dirichlet
N_IBM_BANKS   = 2      # IBM AML split into 2 banks by bank ID
N_BANKS       = N_SAML_BANKS + N_IBM_BANKS   # 6 total
FL_ROUNDS     = 20
LOCAL_EPOCHS  = 5
MLP_CAP       = 100_000
LR            = 0.001
FEDPROX_MU    = 0.01
TEST_FRAC     = 0.15
VAL_FRAC      = 0.15
MIN_FRAUD     = 5
THRESH        = 0.5
AP_KS         = (50, 100, 200)
DIRICHLET_ALPHA = 0.5   # for SAML-D internal partition

# IBM AML: high-risk bank IDs (numeric) mapped to risk flag
IBM_HIGH_RISK_BANKS = {11, 22, 33, 44, 55, 66}  # arbitrary — adjust if needed

# Payment format mapping IBM → unified type
IBM_PAY_MAP = {
    'ACH':          'Bank Transfer',
    'Wire':         'Wire',
    'Cheque':       'Cheque',
    'Credit Card':  'Credit Card',
    'Debit Card':   'Debit Card',
    'Cash':         'Cash',
    'Bitcoin':      'Bitcoin',
    'Reinvestment': 'Reinvestment',
}

HIGH_RISK_COUNTRIES = {'mexico', 'turkey', 'morocco', 'uae', 'iran',
                        'nigeria', 'russia', 'venezuela', 'north korea'}

# ================================================================
# 1. PATHS

cuml.accel installed — sklearn MLP on GPU
All imports OK


## Cell 3: Data Loading Functions

In [3]:
# ================================================================
def find_saml():
    candidates = [
        '/kaggle/input/datasets/berkanoztas/synthetic-transaction-monitoring-dataset-aml/SAML-D.csv',
        '/kaggle/input/synthetic-transaction-monitoring-dataset-aml/SAML-D.csv',
    ]
    for p in candidates:
        if os.path.exists(p): return p
    for root, _, files in os.walk('/kaggle/input'):
        for f in files:
            if f.endswith('.csv') and 'SAML' in f.upper():
                return os.path.join(root, f)
    return None

def find_ibm():
    for root, _, files in os.walk('/kaggle/input'):
        for f in files:
            if f == IBM_FILE:
                return os.path.join(root, f)
    return None

# ================================================================
# 2. LOAD & PREPROCESS SAML-D
# ================================================================
def load_saml():
    path = find_saml()
    if not path: raise FileNotFoundError("SAML-D not found")
    print(f"Loading SAML-D: {path}")
    df = pd.read_csv(path, nrows=SAML_ROWS, low_memory=False)
    df.columns = [c.strip() for c in df.columns]
    print(f"  {len(df):,} rows | fraud: {df['Is_laundering'].sum():,} ({df['Is_laundering'].mean()*100:.3f}%)")
    return df

def preprocess_saml(df):
    y    = df['Is_laundering'].astype(int).values
    typ  = df['Laundering_type'].astype(str).fillna('Unknown').values
    src  = np.full(len(df), 'saml', dtype=object)  # data source tag

    le_t = LabelEncoder(); typ_enc = le_t.fit_transform(typ)
    le_s = LabelEncoder()
    sl_enc = le_s.fit_transform(df['Sender_bank_location'].astype(str).fillna('Unknown'))
    le_r = LabelEncoder()
    rl_enc = le_r.fit_transform(df['Receiver_bank_location'].astype(str).fillna('Unknown'))

    amt = df['Amount'].fillna(0).abs()
    amt_log   = np.log1p(amt).values
    amt_round = (amt % 1000 == 0).astype(int).values
    amt_thresh= (amt < 10000).astype(int).values

    t = pd.to_numeric(df['Time'], errors='coerce').fillna(0)
    hr_sin = np.sin(2 * np.pi * t / 86400).values
    hr_cos = np.cos(2 * np.pi * t / 86400).values

    le_p = LabelEncoder()
    pt_enc = le_p.fit_transform(df['Payment_type'].astype(str).fillna('Unknown'))

    sl_risk = df['Sender_bank_location'].astype(str).str.lower().apply(
        lambda x: int(any(h in x for h in HIGH_RISK_COUNTRIES))).values
    rl_risk = df['Receiver_bank_location'].astype(str).str.lower().apply(
        lambda x: int(any(h in x for h in HIGH_RISK_COUNTRIES))).values
    cross_border = (df['Sender_bank_location'].astype(str) !=
                    df['Receiver_bank_location'].astype(str)).astype(int).values
    curr_mm = (df['Payment_currency'].astype(str) !=
               df['Received_currency'].astype(str)).astype(int).values

    # dataset_source feature: 0 = SAML-D, 1 = IBM AML
    ds_src = np.zeros(len(df), dtype=np.float32)

    X = np.stack([
        amt_log, amt_round, amt_thresh,
        hr_sin, hr_cos, pt_enc.astype(float),
        typ_enc.astype(float), sl_enc.astype(float), rl_enc.astype(float),
        sl_risk, rl_risk, cross_border, curr_mm,
        ds_src
    ], axis=1).astype(np.float32)

    print(f"  SAML-D features: {X.shape}")
    return X, y, typ, src

# ================================================================
# 3. LOAD & PREPROCESS IBM AML
# ================================================================
def load_ibm():
    path = find_ibm()
    if not path: raise FileNotFoundError(f"{IBM_FILE} not found")
    print(f"Loading IBM AML: {path}")
    df = pd.read_csv(path, low_memory=False)
    df.columns = [c.strip() for c in df.columns]
    print(f"  {len(df):,} rows | fraud: {df['Is Laundering'].sum():,} ({df['Is Laundering'].mean()*100:.3f}%)")
    return df

def preprocess_ibm(df):
    """
    Align IBM AML schema to SAML-D feature space.

    IBM AML columns:
        Timestamp, From Bank, Account, To Bank, Account.1,
        Amount Received, Receiving Currency, Amount Paid,
        Payment Currency, Payment Format, Is Laundering

    Mapping:
        Amount Paid        -> amt_log, amt_round, amt_thresh
        Timestamp          -> hr_sin, hr_cos
        Payment Format     -> pt_enc  (mapped to unified categories)
        From Bank (int ID) -> sl_enc  (bank ID as encoded location)
        To Bank (int ID)   -> rl_enc
        From Bank in risk  -> sl_risk (high-risk bank IDs)
        From Bank != To    -> cross_border
        Payment Currency != Receiving Currency -> curr_mm
        Laundering_type    -> 'IBM_Laundering' if fraud, else 'IBM_Normal'
        dataset_source     -> 1 (IBM AML)
    """
    y   = df['Is Laundering'].astype(int).values
    src = np.full(len(df), 'ibm', dtype=object)

    # Synthesise typology labels for IBM AML
    # fraud rows get typology based on Payment Format (best proxy we have)
    # non-fraud rows get 'IBM_Normal'
    pay_fmt = df['Payment Format'].astype(str).fillna('Unknown')
    typ = np.where(
        y == 1,
        ('IBM_' + pay_fmt).values,   # e.g. IBM_Wire, IBM_ACH, IBM_Bitcoin
        'IBM_Normal'
    )

    # Amount
    amt = df['Amount Paid'].fillna(0).abs()
    amt_log   = np.log1p(amt).values.astype(np.float32)
    amt_round = (amt % 1000 == 0).astype(int).values.astype(np.float32)
    amt_thresh= (amt < 10000).astype(int).values.astype(np.float32)

    # Time from Timestamp
    ts = pd.to_datetime(df['Timestamp'], errors='coerce')
    hours = ts.dt.hour.fillna(0).values
    hr_sin = np.sin(2 * np.pi * hours / 24).astype(np.float32)
    hr_cos = np.cos(2 * np.pi * hours / 24).astype(np.float32)

    # Payment format -> unified encoding
    pay_mapped = pay_fmt.map(lambda x: IBM_PAY_MAP.get(x, x))
    le_p = LabelEncoder()
    pt_enc = le_p.fit_transform(pay_mapped).astype(np.float32)

    # Typology encoding
    le_t = LabelEncoder()
    typ_enc = le_t.fit_transform(typ).astype(np.float32)

    # Bank location: use From Bank / To Bank IDs as location proxies
    from_bank = df['From Bank'].fillna(0).astype(str)
    to_bank   = df['To Bank'].fillna(0).astype(str)
    le_s = LabelEncoder()
    sl_enc = le_s.fit_transform(from_bank).astype(np.float32)
    le_r = LabelEncoder()
    rl_enc = le_r.fit_transform(to_bank).astype(np.float32)

    # Risk flags: mark certain bank ID ranges as high-risk
    fb_int = df['From Bank'].fillna(0).astype(int)
    tb_int = df['To Bank'].fillna(0).astype(int)
    sl_risk = fb_int.isin(IBM_HIGH_RISK_BANKS).astype(int).values.astype(np.float32)
    rl_risk = tb_int.isin(IBM_HIGH_RISK_BANKS).astype(int).values.astype(np.float32)

    # Cross-border: different bank IDs
    cross_border = (df['From Bank'].fillna(-1) != df['To Bank'].fillna(-2)).astype(int).values.astype(np.float32)

    # Currency mismatch
    curr_mm = (df['Payment Currency'].astype(str) !=
               df['Receiving Currency'].astype(str)).astype(int).values.astype(np.float32)

    # Dataset source flag: 1 = IBM AML
    ds_src = np.ones(len(df), dtype=np.float32)

    X = np.stack([
        amt_log, amt_round, amt_thresh,
        hr_sin, hr_cos, pt_enc,
        typ_enc, sl_enc, rl_enc,
        sl_risk, rl_risk, cross_border, curr_mm,
        ds_src
    ], axis=1).astype(np.float32)

    print(f"  IBM AML features: {X.shape}")
    return X, y, typ, src


## Cell 4: Merge + Partition

In [4]:
# ================================================================
# 4. MERGE + PARTITION
# ================================================================
def merge_datasets(X_saml, y_saml, typ_saml, src_saml,
                   X_ibm,  y_ibm,  typ_ibm,  src_ibm):
    """
    Stack SAML-D and IBM AML into one merged dataset.
    Standardise features jointly so scales are comparable.
    """
    X   = np.vstack([X_saml, X_ibm])
    y   = np.concatenate([y_saml, y_ibm])
    typ = np.concatenate([typ_saml, typ_ibm])
    src = np.concatenate([src_saml, src_ibm])

    print(f"\nMerged dataset:")
    print(f"  Total rows : {len(X):,}")
    print(f"  Total fraud: {y.sum():,} ({y.mean()*100:.3f}%)")
    print(f"  SAML-D     : {(src=='saml').sum():,} rows | {y[src=='saml'].sum():,} fraud")
    print(f"  IBM AML    : {(src=='ibm').sum():,} rows  | {y[src=='ibm'].sum():,} fraud")
    print(f"  Features   : {X.shape[1]}")
    return X, y, typ, src

def partition_merged(X, y, typ, src):
    """
    Partition strategy: DATA SOURCE partition.
    
    Banks 0-3: SAML-D rows, Dirichlet(alpha=0.5) fraud allocation
    Banks 4-5: IBM AML rows, split by bank ID range (dataset_source feature=1)
    
    This creates genuine typology heterogeneity between bank groups:
    - SAML-D banks see cross-border wire, smurfing, structuring (28 typologies)
    - IBM AML banks see ACH, wire layering, Reinvestment patterns
    """
    np.random.seed(42)

    # Split by source
    saml_idx = np.where(src == 'saml')[0]
    ibm_idx  = np.where(src == 'ibm')[0]

    # ── SAML-D banks (0-3): Dirichlet partition ──
    saml_fi = saml_idx[y[saml_idx] == 1]
    saml_li = saml_idx[y[saml_idx] == 0]
    np.random.shuffle(saml_fi); np.random.shuffle(saml_li)

    props = np.random.dirichlet([DIRICHLET_ALPHA] * N_SAML_BANKS)
    fcnts = (props * len(saml_fi)).astype(int)
    fcnts[-1] = len(saml_fi) - fcnts[:-1].sum()
    fcnts = np.maximum(fcnts, MIN_FRAUD)

    lper = len(saml_li) // N_SAML_BANKS
    banks = []
    fp = lp = 0

    for i in range(N_SAML_BANKS):
        ln  = lper if i < N_SAML_BANKS - 1 else len(saml_li) - lp
        idx = np.concatenate([saml_fi[fp:fp+fcnts[i]], saml_li[lp:lp+ln]])
        np.random.shuffle(idx)
        banks.append(_split(i, X[idx], y[idx], typ[idx], f'SAML-Bank{i}'))
        fp += fcnts[i]; lp += ln

    # ── IBM AML banks (4-5): split by dataset_source variation ──
    # Split IBM rows in half (by index order — approximates bank ID split)
    ibm_fi = ibm_idx[y[ibm_idx] == 1]
    ibm_li = ibm_idx[y[ibm_idx] == 0]
    np.random.shuffle(ibm_fi); np.random.shuffle(ibm_li)

    mid_f = len(ibm_fi) // 2
    mid_l = len(ibm_li) // 2

    for j, (fi_slice, li_slice) in enumerate([
        (ibm_fi[:mid_f],  ibm_li[:mid_l]),
        (ibm_fi[mid_f:],  ibm_li[mid_l:])
    ]):
        bid = N_SAML_BANKS + j
        idx = np.concatenate([fi_slice, li_slice])
        if len(idx) == 0: continue
        np.random.shuffle(idx)
        banks.append(_split(bid, X[idx], y[idx], typ[idx], f'IBM-Bank{j}'))

    print(f"\nPartition — {len(banks)} banks total ({N_SAML_BANKS} SAML-D + {len(banks)-N_SAML_BANKS} IBM AML):")
    for b in banks:
        print(f"  {b['name']:15s}: {b['n_total']:8,} txns | "
              f"{b['n_fraud']:5,} fraud ({b['fraud_frac']*100:.3f}%) | "
              f"source={b['source']}")
    return banks

def _split(bid, X, y, typ, name):
    source = 'ibm' if bid >= N_SAML_BANKS else 'saml'
    strat = y if y.sum() >= 2 else None
    try:
        Xtr,Xte,ytr,yte,ttr,tte = train_test_split(
            X, y, typ, test_size=TEST_FRAC, random_state=42, stratify=strat)
        s2 = ytr if ytr.sum() >= 2 else None
        Xtr,Xvl,ytr,yvl,ttr,tvl = train_test_split(
            Xtr, ytr, ttr,
            test_size=VAL_FRAC/(1-TEST_FRAC), random_state=42, stratify=s2)
    except Exception:
        s = max(2, len(X)//5)
        Xtr,ytr,ttr = X[s:],    y[s:],    typ[s:]
        Xvl,yvl,tvl = X[s//2:s],y[s//2:s],typ[s//2:s]
        Xte,yte,tte = X[:s//2], y[:s//2], typ[:s//2]

    sc  = StandardScaler()
    Xtr = sc.fit_transform(Xtr).astype(np.float32)
    Xvl = sc.transform(Xvl).astype(np.float32)
    Xte = sc.transform(Xte).astype(np.float32)

    return dict(
        id=bid, name=name, source=source,
        X_train=Xtr, y_train=ytr, typ_train=ttr,
        X_val=Xvl,   y_val=yvl,   typ_val=tvl,
        X_test=Xte,  y_test=yte,  typ_test=tte,
        n_fraud=int(y.sum()), n_total=len(y),
        fraud_frac=y.mean(),
    )

def safe_smote(X, y):
    if y.sum() < 5 or len(X) < 20: return X, y
    try:
        k = min(5, int(y.sum()) - 1)
        Xs, ys = SMOTE(random_state=42, k_neighbors=k).fit_resample(X, y)
        return Xs.astype(np.float32), ys.astype(np.int64)
    except Exception:
        return X, y

def get_mlp_data(bank):
    Xtr, ytr = bank['X_train'], bank['y_train']
    if len(Xtr) > MLP_CAP:
        fi = np.where(ytr == 1)[0]
        li = np.where(ytr == 0)[0]
        nf = min(len(fi), MLP_CAP // 10)
        nl = min(len(li), MLP_CAP - nf)
        np.random.shuffle(fi); np.random.shuffle(li)
        idx = np.concatenate([fi[:nf], li[:nl]])
        np.random.shuffle(idx)
        Xtr, ytr = Xtr[idx], ytr[idx]
    return safe_smote(Xtr, ytr)


## Cell 5: MLP + FL Algorithms

In [5]:
# ================================================================
# 5. MLP (cuml.accel)
# ================================================================
def make_mlp():
    return MLPClassifier(
        hidden_layer_sizes=(128, 64, 32), activation='relu',
        solver='adam', learning_rate_init=LR,
        max_iter=LOCAL_EPOCHS, random_state=42,
        warm_start=False, tol=1e-4)

def get_weights(m):
    return [c.copy() for c in m.coefs_] + [i.copy() for i in m.intercepts_]

def set_weights(m, w):
    n = len(m.coefs_)
    m.coefs_ = [x.copy() for x in w[:n]]
    m.intercepts_ = [x.copy() for x in w[n:]]

def init_mlp(X, y):
    if 0 not in y: X,y = np.vstack([X,X[0:1]]), np.append(y,0)
    if 1 not in y: X,y = np.vstack([X,X[0:1]]), np.append(y,1)
    m = make_mlp(); m.fit(X, y); return m

def clone_mlp(template, w):
    m = make_mlp()
    tiny = np.zeros((2, template.coefs_[0].shape[0]), dtype=np.float32)
    try: m.fit(tiny, np.array([0,1]))
    except: pass
    set_weights(m, w); return m

def mlp_proba(m, X):
    try:    return m.predict_proba(X)[:,1]
    except: return np.full(len(X), 0.5)

# ================================================================
# 6. FL ALGORITHMS
# ================================================================
def fedavg_agg(gw, wlist, sizes):
    total = sum(sizes)
    return [sum((s/total)*wl[i] for wl,s in zip(wlist,sizes))
            for i in range(len(gw))]

def run_fedavg(banks, gw, tmpl):
    wl=[]; sz=[]
    for b in banks:
        Xtr,ytr=get_mlp_data(b); lm=clone_mlp(tmpl,gw); lm.fit(Xtr,ytr)
        wl.append(get_weights(lm)); sz.append(len(Xtr)); del lm; flush()
    return fedavg_agg(gw,wl,sz)

def run_fedprox(banks, gw, tmpl):
    wl=[]; sz=[]
    for b in banks:
        Xtr,ytr=get_mlp_data(b); lm=clone_mlp(tmpl,gw); lm.fit(Xtr,ytr)
        local_w=get_weights(lm)
        prox=[lw-FEDPROX_MU*(lw-g) for lw,g in zip(local_w,gw)]
        wl.append(prox); sz.append(len(Xtr)); del lm; flush()
    return fedavg_agg(gw,wl,sz)

def run_fednova(banks, gw, tmpl):
    dl=[]; sz=[]; st=[]
    for b in banks:
        Xtr,ytr=get_mlp_data(b); lm=clone_mlp(tmpl,gw); lm.fit(Xtr,ytr)
        lw=get_weights(lm); delta=[l-g for l,g in zip(lw,gw)]
        tau=LOCAL_EPOCHS*max(len(Xtr)//32,1)
        dl.append(delta); sz.append(len(Xtr)); st.append(tau); del lm; flush()
    total=sum(sz); tau_g=sum(t*s/total for t,s in zip(st,sz))
    return [gw[i]+tau_g*sum((s/total)*(d[i]/max(t,1))
            for d,s,t in zip(dl,sz,st)) for i in range(len(gw))]

def run_persfl(banks, gw, tmpl, bw):
    BLEND=0.5; wl=[]; sz=[]
    for b in banks:
        bid=b['id']; Xtr,ytr=get_mlp_data(b)
        lm=clone_mlp(tmpl,bw.get(bid,gw)); lm.fit(Xtr,ytr)
        lw=get_weights(lm)
        bw[bid]=[BLEND*l+(1-BLEND)*g for l,g in zip(lw,gw)]
        wl.append(lw); sz.append(len(Xtr)); del lm; flush()
    return fedavg_agg(gw,wl,sz), bw

def run_fl(banks, strategy, n_feat):
    print(f"  [{strategy.upper()}] {FL_ROUNDS} rounds", end='  ')
    init_X = np.vstack([b['X_train'][:50] for b in banks])
    init_y = np.concatenate([b['y_train'][:50] for b in banks])
    tmpl = init_mlp(init_X, init_y)
    gw = get_weights(tmpl); bw = {}; hist = []

    for rnd in range(1, FL_ROUNDS+1):
        if strategy=='fedavg':   gw=run_fedavg(banks,gw,tmpl)
        elif strategy=='fedprox': gw=run_fedprox(banks,gw,tmpl)
        elif strategy=='fednova': gw=run_fednova(banks,gw,tmpl)
        elif strategy=='persfl':  gw,bw=run_persfl(banks,gw,tmpl,bw)

        if rnd%5==0 or rnd==FL_ROUNDS:
            print(f"{rnd}", end=' ', flush=True)
            gm=clone_mlp(tmpl,gw)
            for b in banks:
                yt=b['y_test']
                if yt.sum()==0: continue
                em=clone_mlp(tmpl,bw[b['id']]) if strategy=='persfl' and b['id'] in bw else gm
                probs=mlp_proba(em,b['X_test']); preds=(probs>=THRESH).astype(int)
                hist.append(dict(round=rnd, bank_id=b['id'], bank_name=b['name'],
                    strategy=strategy, source=b['source'],
                    f1=f1_score(yt,preds,zero_division=0),
                    auprc=average_precision_score(yt,probs)))
        flush()
    print()
    return clone_mlp(tmpl,gw), bw, hist


## Cell 6: Local Experts + MoE Gates + Metrics

In [6]:
# ================================================================
# 7. LOCAL EXPERTS
# ================================================================
class XGBExpert:
    name='xgb'
    def __init__(self,pw):
        self.m=xgb.XGBClassifier(n_estimators=200,max_depth=6,
            scale_pos_weight=pw,use_label_encoder=False,eval_metric='aucpr',
            subsample=0.8,colsample_bytree=0.8,device='cuda',
            random_state=42,verbosity=0); self.trained=False
    def fit(self,X,y): self.m.fit(X,y); self.trained=True
    def predict_proba(self,X):
        return self.m.predict_proba(X)[:,1] if self.trained else np.full(len(X),0.5)

class LGBMExpert:
    name='lgbm'
    def __init__(self,pw):
        self.m=lgb.LGBMClassifier(n_estimators=200,max_depth=6,
            scale_pos_weight=pw,subsample=0.8,colsample_bytree=0.8,
            device='cpu',random_state=42,verbose=-1); self.trained=False
    def fit(self,X,y): self.m.fit(X,y); self.trained=True
    def predict_proba(self,X):
        return self.m.predict_proba(X)[:,1] if self.trained else np.full(len(X),0.5)

def train_experts(banks):
    experts={}
    for b in banks:
        bid=b['id']; experts[bid]={}
        Xtr,ytr=safe_smote(b['X_train'],b['y_train'])
        n0=(ytr==0).sum(); n1=max((ytr==1).sum(),1); pw=float(n0/n1)
        for Cls in [XGBExpert,LGBMExpert]:
            m=Cls(pw); m.fit(Xtr,ytr)
            p=m.predict_proba(b['X_test']); pd_=(p>=THRESH).astype(int)
            f1=f1_score(b['y_test'],pd_,zero_division=0)
            ap=average_precision_score(b['y_test'],p)
            print(f"  {b['name']:15s} {m.name.upper():5s}: F1={f1:.4f}  AUPRC={ap:.4f}")
            experts[bid][m.name]=m
    return experts

# ================================================================
# 8. MoE GATES
# ================================================================
class StaticGate:
    name='moe_static'
    def get_weights(self,n,avail,**kw):
        w=np.where(avail,1.,0.); return w/w.sum() if w.sum()>0 else w
    def update(self,*a,**k): pass

class PerformanceGate:
    name='moe_performance'
    def __init__(self): self._w={}
    def update(self,bid,vf1s,avail):
        s=np.where(avail,np.clip(vf1s,0,None),0.)
        self._w[bid]=s/s.sum() if s.sum()>0 else np.ones(len(s))/len(s)
    def get_weights(self,n,avail,bid=None,**kw):
        return self._w.get(bid,np.ones(n)/n)

class TypologyAwareGate:
    name='moe_typology_aware'
    def __init__(self):
        self._tbl=defaultdict(lambda:defaultdict(dict)); self._fb={}
    def update(self,bid,typ_f1s,avail,val_f1s):
        for ei,(tf,av) in enumerate(zip(typ_f1s,avail)):
            if av: self._tbl[bid][ei]=tf
        s=np.where(avail,np.clip(val_f1s,0,None),0.)
        self._fb[bid]=s/s.sum() if s.sum()>0 else np.ones(len(s))/len(s)
    def get_weights(self,n,avail,bid=None,txn_typ=None,**kw):
        if txn_typ is None or bid not in self._tbl:
            return self._fb.get(bid,np.ones(n)/n)
        w=np.zeros(n)
        for ei in range(n):
            if avail[ei] and ei in self._tbl[bid]:
                w[ei]=self._tbl[bid][ei].get(txn_typ,0.)
        return w/w.sum() if w.sum()>0 else self._fb.get(bid,np.ones(n)/n)

def _typ_f1(y_val,probs,typ_val):
    preds=(probs>=THRESH).astype(int); out={}
    for t in np.unique(typ_val):
        mask=typ_val==t
        if mask.sum()<2 or y_val[mask].sum()==0: continue
        out[t]=f1_score(y_val[mask],preds[mask],zero_division=0)
    return out

# ================================================================
# 9. METRICS
# ================================================================
TYP_W={
    'Cycle':2.5,'Bipartite':2.5,'Stacked Bipartite':2.5,
    'Structuring':2.5,'Smurfing':2.0,'Scatter-Gather':2.0,
    'Gather-Scatter':2.0,'Fan_Out':2.0,'Fan_In':2.0,
    'Layered_Fan_Out':2.0,'Layered_Fan_In':2.0,
    'Deposit-Send':2.0,'Over-Invoicing':2.0,
    'IBM_Wire':2.5,'IBM_ACH':2.0,'IBM_Bitcoin':2.5,
    'IBM_Cheque':1.5,'IBM_Cash':1.5,'IBM_Credit Card':2.0,
    'IBM_Reinvestment':2.0,
}

def compute_metrics(y_true,probs,typ=None):
    if y_true.sum()==0: return _em()
    preds=(probs>=THRESH).astype(int)
    f1=f1_score(y_true,preds,zero_division=0)
    prec=precision_score(y_true,preds,zero_division=0)
    rec=recall_score(y_true,preds,zero_division=0)
    try:    auprc=average_precision_score(y_true,probs)
    except: auprc=float(y_true.mean())
    mcc=matthews_corrcoef(y_true,preds) if preds.sum()>0 else 0.
    b2=4; d=b2*prec+rec; f2=(1+b2)*prec*rec/d if d>0 else 0.
    sidx=np.argsort(probs)[::-1]
    apk={f'ap_at_{k}':float(y_true[sidx[:min(k,len(y_true))]].sum()/
                             max(min(k,len(y_true)),1)) for k in AP_KS}
    typ_cov=0.; typ_wf1=0.
    if typ is not None:
        laund=[t for t in np.unique(typ) if 'Normal' not in t and t!='Unknown']
        if laund:
            typ_cov=sum(1 for t in laund
                if ((typ==t)&(y_true==1)&(preds==1)).sum()>0)/len(laund)
        ws=wt=0.
        for t in np.unique(typ):
            mask=typ==t
            if mask.sum()<2 or y_true[mask].sum()==0: continue
            f=f1_score(y_true[mask],preds[mask],zero_division=0)
            w=TYP_W.get(t,1.5 if 'Normal' not in t else 0.)
            ws+=w*f; wt+=w
        typ_wf1=ws/max(wt,1e-8)
    return {'f1':f1,'precision':prec,'recall':rec,'auprc':auprc,
            'mcc':mcc,'f2':f2,**apk,'typ_coverage':typ_cov,'typ_wf1':typ_wf1}

def _em():
    return {k:0. for k in ['f1','precision','recall','auprc','mcc','f2',
                            'ap_at_50','ap_at_100','ap_at_200',
                            'typ_coverage','typ_wf1']}

def agg(ml):
    if not ml: return {}
    return {k:round(float(np.mean([m[k] for m in ml])),4) for k in ml[0]}

def fairness(f1s,lf=None):
    r=dict(client_equity=round(float(np.std(f1s)),4),
           worst_bank_f1=round(float(min(f1s)),4),
           best_bank_f1=round(float(max(f1s)),4))
    if lf and len(lf)==len(f1s):
        r['collab_gain']=round(float(np.mean([a-b for a,b in zip(f1s,lf)])),4)
    return r


## Cell 7: Evaluation + Plots

In [7]:
# ================================================================
# 10. EVALUATION
# ================================================================
def evaluate_all(banks, fl_results, experts):
    rows=[]; n_exp=len(fl_results)+2

    loc_f1={}
    for b in banks:
        bid=b['id']; yt=b['y_test']
        if yt.sum()==0: continue
        best=0.
        for en in ['xgb','lgbm']:
            m=experts[bid].get(en)
            if m and m.trained:
                p=m.predict_proba(b['X_test'])
                best=max(best,f1_score(yt,(p>=THRESH).astype(int),zero_division=0))
        loc_f1[bid]=best

    def _row(strat,mtype,bm,lf=None,extra={}):
        a=agg(bm); fa=fairness([m['f1'] for m in bm],lf)
        return {'strategy':strat,'model_type':mtype,**a,**fa,**extra}

    FL_STRATS = list(fl_results.keys())

    for strat,(fl_model,bw,_) in fl_results.items():
        bm=[]; lf=[]
        for b in banks:
            yt=b['y_test']; tte=b.get('typ_test')
            if yt.sum()==0: continue
            if strat=='persfl' and b['id'] in bw:
                probs=mlp_proba(bw[b['id']],b['X_test'])
            else:
                probs=mlp_proba(fl_model,b['X_test'])
            bm.append(compute_metrics(yt,probs,tte))
            lf.append(loc_f1.get(b['id'],0))
        rows.append(_row(strat,'fl',bm,lf))

    for en in ['xgb','lgbm']:
        bm=[]
        for b in banks:
            yt=b['y_test']; tte=b.get('typ_test')
            if yt.sum()==0: continue
            m=experts[b['id']].get(en)
            probs=m.predict_proba(b['X_test']) if (m and m.trained) else np.full(len(yt),0.5)
            bm.append(compute_metrics(yt,probs,tte))
        rows.append(_row(en,'local_expert',bm))

    for gate in [StaticGate(),PerformanceGate(),TypologyAwareGate()]:
        bm=[]; lf=[]
        for b in banks:
            bid=b['id']; yt=b['y_test']
            tte=b.get('typ_test',np.array(['Unknown']*len(yt)))
            tvl=b.get('typ_val', np.array(['Unknown']*len(b['y_val'])))
            if yt.sum()==0: continue

            tps=[]; vf1s=[]; avl=[]; tyf=[]
            for strat,(fl_model,bw,_) in fl_results.items():
                if strat=='persfl' and bid in bw:
                    pv=mlp_proba(bw[bid],b['X_val']); pt=mlp_proba(bw[bid],b['X_test'])
                else:
                    pv=mlp_proba(fl_model,b['X_val']); pt=mlp_proba(fl_model,b['X_test'])
                vf1s.append(f1_score(b['y_val'],(pv>=THRESH).astype(int),zero_division=0))
                tyf.append(_typ_f1(b['y_val'],pv,tvl))
                avl.append(True); tps.append(pt)

            for en in ['xgb','lgbm']:
                m=experts[bid].get(en)
                if m and m.trained:
                    pv=m.predict_proba(b['X_val']); pt=m.predict_proba(b['X_test'])
                    vf1s.append(f1_score(b['y_val'],(pv>=THRESH).astype(int),zero_division=0))
                    tyf.append(_typ_f1(b['y_val'],pv,tvl))
                    avl.append(True); tps.append(pt)
                else:
                    vf1s.append(0.); tyf.append({}); avl.append(False)
                    tps.append(np.full(len(yt),0.5))

            avl=np.array(avl); vf1s=np.array(vf1s)

            if isinstance(gate,TypologyAwareGate):
                gate.update(bid,tyf,avl,val_f1s=vf1s)
                ap=np.zeros(len(yt))
                for ut in np.unique(tte):
                    mask=tte==ut
                    w=gate.get_weights(n_exp,avl,bid=bid,txn_typ=ut)
                    for i,pt in enumerate(tps): ap[mask]+=w[i]*pt[mask]
            elif isinstance(gate,PerformanceGate):
                gate.update(bid,vf1s,avl)
                w=gate.get_weights(n_exp,avl,bid=bid)
                ap=sum(w[i]*tps[i] for i in range(n_exp))
            else:
                w=gate.get_weights(n_exp,avl)
                ap=sum(w[i]*tps[i] for i in range(n_exp))

            bm.append(compute_metrics(yt,ap,tte))
            lf.append(loc_f1.get(bid,0))

        rows.append(_row(gate.name,'moe',bm,lf))
    return rows

# ================================================================
# 11. PLOTS
# ================================================================
COLORS={
    'fedavg':'#4F46A0','fedprox':'#0F6E56','fednova':'#854F0B','persfl':'#185FA5',
    'xgb':'#A32D2D','lgbm':'#D85A30',
    'moe_static':'#AAAAAA','moe_performance':'#1D9E75','moe_typology_aware':'#9333EA',
}

def plot_results(df_bm, fl_hist):
    strats=sorted(df_bm['strategy'].unique())
    fig,axes=plt.subplots(1,3,figsize=(18,5.5))
    fig.suptitle('SAML-D + IBM AML Merged Benchmark — Data-Source Partition\n'
                 'MLP FL + XGB + LGBM + MoE', fontsize=11, fontweight='bold')

    ax=axes[0]; x=np.arange(len(strats)); w=0.22
    for mi,(m,lbl,c) in enumerate([('auprc','AUPRC*','#2E4057'),
                                    ('f2','F2*','#048A81'),('f1','F1','#54C6EB')]):
        vals=[float(df_bm[df_bm['strategy']==s][m].mean()) for s in strats]
        ax.bar(x+(mi-1)*w,vals,w,label=lbl,color=c,alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([s.replace('moe_','MoE-') for s in strats],fontsize=7,rotation=40,ha='right')
    ax.set_ylim(0,1); ax.legend(fontsize=8,frameon=False)
    ax.set_title('Primary Metrics',fontsize=9,fontweight='bold')
    ax.spines[['top','right']].set_visible(False); ax.grid(axis='y',alpha=0.25)

    ax=axes[1]
    for si,s in enumerate(strats):
        c=COLORS.get(s,'#999')
        ap=float(df_bm[df_bm['strategy']==s]['auprc'].mean())
        rc=float(df_bm[df_bm['strategy']==s]['recall'].mean())
        ax.bar(si-0.18,ap,0.35,color=c,alpha=0.85)
        ax.bar(si+0.18,rc,0.35,color=c,alpha=0.4,edgecolor=c,linewidth=1.5)
    ax.set_xticks(range(len(strats)))
    ax.set_xticklabels([s.replace('moe_','MoE-') for s in strats],fontsize=7,rotation=40,ha='right')
    ax.set_ylim(0,1); ax.set_title('AUPRC vs Recall',fontsize=9,fontweight='bold')
    ax.spines[['top','right']].set_visible(False); ax.grid(axis='y',alpha=0.25)

    ax=axes[2]
    dfh=pd.DataFrame(fl_hist)
    if not dfh.empty:
        for strat in ['fedavg','fedprox','persfl']:
            sub=dfh[dfh['strategy']==strat]
            if sub.empty: continue
            g=sub.groupby('round')['auprc'].mean().reset_index()
            ax.plot(g['round'],g['auprc'],label=strat.upper(),
                    color=COLORS.get(strat,'#999'),lw=2,marker='o',ms=4)
    ax.set_xlabel('FL Round'); ax.set_ylim(0,1)
    ax.set_title('FL Learning Curves',fontsize=9,fontweight='bold')
    ax.legend(fontsize=8,frameon=False)
    ax.spines[['top','right']].set_visible(False); ax.grid(alpha=0.25)

    plt.tight_layout()
    p=f'{OUT}/merged_benchmark_results.png'
    plt.savefig(p,dpi=150,bbox_inches='tight'); plt.close()
    print(f"Saved: {p}")

def print_summary(df_bm):
    print("\n"+"="*70)
    print("MERGED BENCHMARK — SAML-D + IBM AML (Data-Source Partition)")
    print("PRIMARY: AUPRC, MCC, F2")
    print("="*70)
    cols=['strategy','auprc','mcc','f2','f1','recall',
          'ap_at_100','typ_coverage','typ_wf1',
          'client_equity','worst_bank_f1','collab_gain']
    avail=[c for c in cols if c in df_bm.columns]
    print(df_bm[avail].sort_values('auprc',ascending=False).to_string(index=False))

    print("\nBEST (AUPRC):")
    best=df_bm.loc[df_bm['auprc'].idxmax()]
    print(f"  {best['strategy']:25s}  AUPRC={best['auprc']:.4f}  "
          f"MCC={best['mcc']:.4f}  F2={best['f2']:.4f}")

# ================================================================
# MAIN

## Cell 8: Run Everything

In [8]:
# ================================================================
print(f"\n{'='*60}")
print("MERGED: SAML-D + IBM AML — FL + MoE (GPU)")
print(f"{'='*60}\n")

# Load both datasets
df_saml = load_saml()
df_ibm  = load_ibm()

# Preprocess
print("\nPreprocessing SAML-D...")
X_saml, y_saml, typ_saml, src_saml = preprocess_saml(df_saml)
del df_saml; flush()

print("Preprocessing IBM AML...")
X_ibm, y_ibm, typ_ibm, src_ibm = preprocess_ibm(df_ibm)
del df_ibm; flush()

print(f"Preprocessing done | {elapsed()}")

# Merge
X, y, typ, src = merge_datasets(X_saml, y_saml, typ_saml, src_saml,
                                 X_ibm,  y_ibm,  typ_ibm,  src_ibm)
del X_saml, y_saml, typ_saml, src_saml
del X_ibm,  y_ibm,  typ_ibm,  src_ibm
flush()

# Partition
banks = partition_merged(X, y, typ, src)

# FL Training
fl_results={}; all_hist=[]
print(f"\nFL training ({N_BANKS} banks, GPU via cuml.accel):")
for strat in ['fedavg','fedprox','persfl']:
    fl_model,bw,hist=run_fl(banks,strat,X.shape[1])
    if strat=='persfl':
        init_X=np.vstack([b['X_train'][:50] for b in banks])
        init_y=np.concatenate([b['y_train'][:50] for b in banks])
        tm=init_mlp(init_X,init_y)
        bw_m={bid:clone_mlp(tm,w) for bid,w in bw.items()}
        fl_results[strat]=(fl_model,bw_m,hist)
    else:
        fl_results[strat]=(fl_model,{},hist)
    all_hist.extend(hist); flush()

# Local experts
print(f"\nLocal experts (XGB cuda + LGBM cpu)...")
experts=train_experts(banks); flush()

# Evaluate
print(f"\nEvaluating all models + MoE gates...")
bm_rows=evaluate_all(banks,fl_results,experts)

# Save
df_bm=pd.DataFrame(bm_rows)
df_bm.to_csv(f'{OUT}/merged_benchmark.csv',index=False)
pd.DataFrame(all_hist).to_csv(f'{OUT}/merged_fl_history.csv',index=False)

print_summary(df_bm)
plot_results(df_bm, all_hist)

print(f"\nTotal runtime: {elapsed()}")
print(f"\nOutputs:")
for f in sorted(os.listdir(OUT)):
    if f.endswith(('.csv','.png')):
        print(f"  {f}")



MERGED: SAML-D + IBM AML — FL + MoE (GPU)

Loading SAML-D: /kaggle/input/datasets/berkanoztas/synthetic-transaction-monitoring-dataset-aml/SAML-D.csv
  9,500,000 rows | fraud: 9,869 (0.104%)
Loading IBM AML: /kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/HI-Small_Trans.csv
  5,078,345 rows | fraud: 5,177 (0.102%)

Preprocessing SAML-D...
  SAML-D features: (9500000, 14)
Preprocessing IBM AML...
  IBM AML features: (5078345, 14)
Preprocessing done | 1.7m

Merged dataset:
  Total rows : 14,578,345
  Total fraud: 15,046 (0.103%)
  SAML-D     : 9,500,000 rows | 9,869 fraud
  IBM AML    : 5,078,345 rows  | 5,177 fraud
  Features   : 14

Partition — 6 banks total (4 SAML-D + 2 IBM AML):
  SAML-Bank0     : 2,374,558 txns | 2,026 fraud (0.085%) | source=saml
  SAML-Bank1     : 2,377,414 txns | 4,882 fraud (0.205%) | source=saml
  SAML-Bank2     : 2,375,347 txns | 2,815 fraud (0.119%) | source=saml
  SAML-Bank3     : 2,372,681 txns |   146 fraud (0.006%) | so